# Day 194 — Final Capstone Part 3: Model / RAG Layer

**Month 12 · Day 3 of Final Capstone (Days 192–196)**

Day 193 delivered `df_clean` — 400 rows, plan_type normalized to 3 canonical values, 9 duplicates
dropped, `satisfaction_score` correctly left as NaN for the 106 unresolved complaints. Today builds
the two layers that make this capstone more than a cleaning exercise:

| Layer | Tech | Source |
|---|---|---|
| **Layer 3 — ML/Feature Layer** | scikit-learn `TfidfVectorizer` + `LogisticRegression` | Months 6–7, 9 |
| **Layer 4 — RAG/LLM Layer** | LangChain + FAISS + Groq | Month 10 |

**Hard rule carried from Day 193:** only `df_clean` is a valid input to these layers. Never re-read
`df_raw` here — that would silently reintroduce the 9 duplicates and the casing bug into the model
and the retrieval index.


## Concept Notes — Two Different Jobs, One Dataset

`resolved` (True/False) is the only label present for **every** row — `satisfaction_score` only
exists for the 294 resolved cases, so it can't be a supervised target for the full dataset without
reintroducing the MNAR problem Day 193 already fixed. That makes `resolved` the ML target today.

**Layer 3 (ML) asks:** "Can structured features + complaint text predict whether a new complaint
will be resolved?" — a **classification** question, answered once, offline, and monitored.

**Layer 4 (RAG) asks:** "For *this specific* complaint, what did we do for similar complaints
before, and how did that turn out?" — a **retrieval** question, answered fresh per query, using the
294 resolved complaints as a searchable knowledge base with real outcomes (`days_to_resolve`,
`satisfaction_score`) attached.

These are not redundant. A classifier gives one number for the whole population. RAG gives a
grounded, case-specific answer a support agent can actually act on. Confusing the two — expecting
a classifier to "explain" a case, or expecting RAG to give a population-level accuracy number — is
the most common architecture mistake at this stage.


---
## Section 1 — Raw Data + Day 193 Cleaning (Locked, DO NOT MODIFY)

Identical generator to Days 192–193 (seed=192), followed by the Day 193 cleaning pipeline
(casing normalization + de-duplication). Reproduced here only so this notebook runs standalone —
the logic must not change from what Day 193 already tested and locked.


In [1]:
# ============================================================
# RAW DATA GENERATOR + DAY 193 CLEANING PIPELINE — DO NOT MODIFY
# Locked for the entire Month 12 Final Capstone arc (Days 192-196)
# ============================================================
import pandas as pd
import numpy as np
import random

SEED = 192
random.seed(SEED)
np.random.seed(SEED)

N = 400

plan_types_clean = ["Prepaid", "Postpaid", "Broadband"]
plan_types_dirty = ["prepaid", "POSTPAID", "Broadband "]

complaint_templates = [
    "Network signal keeps dropping in {area} area since last week",
    "Billed amount does not match my selected {plan} plan this month",
    "Internet speed far below promised {plan} plan speed during evening hours",
    "Recharge done but plan benefits not activated for 2 days",
    "Customer care call disconnected three times without resolving {issue}",
    "Data balance deducted incorrectly after using app for {issue}",
    "Broadband router keeps restarting every few hours in {area}",
    "Requested plan downgrade last month but still billed for old plan",
    "SMS service not working since the last recharge cycle",
    "Roaming charges applied despite being within home {area} circle",
]
areas = ["Sonipat", "Gurugram", "Panipat", "Rohtak", "Karnal", "Delhi NCR"]
issues = ["video streaming", "UPI payment", "OTP delivery", "customer app login"]

rows = []
for i in range(N):
    cust_id = f"TS{1000+i}"
    plan = random.choice(plan_types_clean)
    tenure = int(np.clip(np.random.exponential(scale=18), 1, 96))
    if plan == "Prepaid":
        bill = round(np.random.normal(249, 60), 2)
    elif plan == "Postpaid":
        bill = round(np.random.normal(599, 120), 2)
    else:
        bill = round(np.random.normal(999, 180), 2)
    bill = max(bill, 99)

    template = random.choice(complaint_templates)
    complaint = template.format(area=random.choice(areas), plan=plan,
                                 issue=random.choice(issues))

    resolved = random.random() < 0.72
    days_to_resolve = int(np.clip(np.random.exponential(scale=2.5), 0, 14)) if resolved else np.nan
    satisfaction = None
    if resolved:
        base = 5 - (days_to_resolve / 4)
        satisfaction = int(np.clip(round(base + np.random.normal(0, 0.6)), 1, 5))

    signup_month_offset = tenure
    signup_date = pd.Timestamp("2026-07-06") - pd.DateOffset(months=int(signup_month_offset))

    rows.append({
        "customer_id": cust_id,
        "plan_type": plan,
        "monthly_bill_inr": bill,
        "tenure_months": tenure,
        "signup_date": signup_date.strftime("%Y-%m-%d"),
        "complaint_text": complaint,
        "resolved": resolved,
        "days_to_resolve": days_to_resolve,
        "satisfaction_score": satisfaction,
    })

df_raw = pd.DataFrame(rows)

dirty_idx = np.random.choice(df_raw.index, size=14, replace=False)
for j, idx in enumerate(dirty_idx):
    df_raw.loc[idx, "plan_type"] = plan_types_dirty[j % len(plan_types_dirty)]

dup_rows = df_raw.sample(n=9, random_state=SEED)
df_raw = pd.concat([df_raw, dup_rows], ignore_index=True)

# --- Day 193 cleaning pipeline (locked) ---
df_clean = df_raw.copy()
df_clean["plan_type"] = df_clean["plan_type"].str.strip().str.capitalize()
df_clean = df_clean.drop_duplicates().reset_index(drop=True)

print(f"df_raw:   {len(df_raw)} rows")
print(f"df_clean: {len(df_clean)} rows, plan_type values: {sorted(df_clean['plan_type'].unique())}")
print(f"Missing satisfaction_score in df_clean: {df_clean['satisfaction_score'].isna().sum()}")
df_clean.head()


df_raw:   409 rows
df_clean: 400 rows, plan_type values: ['Broadband', 'Postpaid', 'Prepaid']
Missing satisfaction_score in df_clean: 106


,customer_id,plan_type,monthly_bill_inr,tenure_months,signup_date,complaint_text,resolved,days_to_resolve,satisfaction_score
0,TS1000,Postpaid,469.76,25,2024-06-06,Customer care call disconnected three times wi...,True,6.0,2.0
1,TS1001,Postpaid,376.93,1,2026-06-06,Data balance deducted incorrectly after using ...,True,0.0,5.0
2,TS1002,Postpaid,653.68,19,2024-12-06,Internet speed far below promised Postpaid pla...,True,3.0,4.0
3,TS1003,Prepaid,205.16,21,2024-10-06,Customer care call disconnected three times wi...,False,NaN,NaN
4,TS1004,Postpaid,495.01,62,2021-05-06,Data balance deducted incorrectly after using ...,True,3.0,4.0


---
## Section 2 — Practice Tasks

### Task 1 — ML / Feature Layer: Predict `resolved` (20 pts)

Build features from `df_clean` and train a baseline classifier for the `resolved` label.

**Why `resolved` and not `satisfaction_score`:** `satisfaction_score` is NaN for 106 rows by
design (Day 193's MNAR finding). Training on it would mean silently dropping those 106 rows or
imputing a fabricated opinion — both violate Day 192/193's locked decisions. `resolved` is present
for all 400 rows, so it is the only label that doesn't reopen an already-closed data-integrity
question.

Steps:
1. `TfidfVectorizer(max_features=100, stop_words='english')` on `complaint_text`
2. One-hot encode `plan_type`, scale `tenure_months` and `monthly_bill_inr` with `StandardScaler`
3. Combine into one feature matrix with `scipy.sparse.hstack`
4. `train_test_split(test_size=0.25, random_state=192, stratify=y)`
5. Train **two** `LogisticRegression` models: one plain, one with `class_weight='balanced'`
6. Print accuracy and F1 for both, plus the majority-class baseline accuracy


In [2]:
# TASK 1: ML Feature Layer
# Goal: train a classifier to predict 'resolved' using TF-IDF + numeric features.
# Method: combine text and structured features, train plain and balanced logistic regression,
#         and print accuracy, F1, and baseline.

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
from scipy.sparse import hstack, csr_matrix
import numpy as np
import pandas as pd

# 1. Text features
tfidf = TfidfVectorizer(max_features=100, stop_words='english')
X_text = tfidf.fit_transform(df_clean['complaint_text'])

# 2. Structured features
plan_dummies = pd.get_dummies(df_clean['plan_type'], prefix='plan')
X_numeric = pd.concat([df_clean[['tenure_months', 'monthly_bill_inr']], plan_dummies], axis=1)
scaler = StandardScaler()
X_numeric_scaled = scaler.fit_transform(X_numeric)

# 3. Combine
X_full = hstack([X_text, csr_matrix(X_numeric_scaled)])
y = df_clean['resolved'].astype(int)

# 4. Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X_full, y, test_size=0.25, random_state=192, stratify=y
)

# 5. Models
clf_plain = LogisticRegression(max_iter=1000, random_state=192)
clf_balanced = LogisticRegression(max_iter=1000, random_state=192, class_weight='balanced')
clf_plain.fit(X_train, y_train)
clf_balanced.fit(X_train, y_train)

# 6. Predictions and metrics
y_pred_plain = clf_plain.predict(X_test)
y_pred_balanced = clf_balanced.predict(X_test)

majority_baseline = max(y_test.mean(), 1 - y_test.mean())

print("Train size:", X_train.shape[0])
print("Test size:", X_test.shape[0])
print("Majority-class baseline accuracy:", majority_baseline)
print("Plain LogisticRegression accuracy:", accuracy_score(y_test, y_pred_plain))
print("Balanced LogisticRegression accuracy:", accuracy_score(y_test, y_pred_balanced))
print("Plain LogisticRegression F1:", f1_score(y_test, y_pred_plain))
print("Balanced LogisticRegression F1:", f1_score(y_test, y_pred_balanced))
print("Balanced confusion matrix:", confusion_matrix(y_test, y_pred_balanced))

Train size: 300
Test size: 100
Majority-class baseline accuracy: 0.73
Plain LogisticRegression accuracy: 0.73
Balanced LogisticRegression accuracy: 0.5
Plain LogisticRegression F1: 0.8439306358381503
Balanced LogisticRegression F1: 0.609375
Balanced confusion matrix: [[11 16]
 [34 39]]


### Task 2 — NRA: Model Business Case (20 pts)

Before writing Reason, check the **actual generator code** in Section 1 for what determines
`resolved`. Do not guess a plausible-sounding mechanism — the same mistake flagged on Day 189
(fabricated carrier mechanism) applies here if you invent a text/feature relationship that isn't
in the code.

Write exactly 2 NRA bullets using only Task 1's printed output.


In [8]:
# TASK 2: NRA Model Business Case
# Goal: write two NRA bullets using numbers from Task 1's output.
# Method: read the generator code to find that 'resolved' is purely random (p=0.72),
#         then reason about why models underperform.

# Retrieve metrics from Task 1 (store in variables for use here)
baseline_acc = majority_baseline
plain_acc = accuracy_score(y_test, y_pred_plain)
bal_acc = accuracy_score(y_test, y_pred_balanced)
plain_f1 = f1_score(y_test, y_pred_plain)
bal_f1 = f1_score(y_test, y_pred_balanced)
cm = confusion_matrix(y_test, y_pred_balanced)
tn, fp, fn, tp = cm.ravel()  # assuming binary

nra_model_bullet_1 = {
    "number": f"Plain model accuracy: {plain_acc:.3f}, Balanced model accuracy: {bal_acc:.3f}, Baseline: {baseline_acc:.3f}",
    "reason": "The generator defines `resolved = random.random() < 0.72` – the label is independent of any feature. No model can learn a real signal.",
    "action": "Do not use this ML classifier for any business decision. Instead, rely on the RAG layer for case-specific, grounded answers."
}

nra_model_bullet_2 = {
    "number": f"Balanced confusion matrix: TN={tn}, FP={fp}, FN={fn}, TP={tp}",
    "reason": "`class_weight='balanced'` forces the model to predict the minority class (unresolved) more often; because the label is purely random, this shift trades correct majority guesses for wrong minority predictions – hence accuracy drops to 0.5.",
    "action": "If this model is revisited with better features, set a minimum performance requirement: it must beat the baseline accuracy by at least 0.05 on a validation set. This threshold will be reviewed by the data science lead on Day 196 and re‑evaluated quarterly if new feature engineering is performed."
}

for i, b in enumerate([nra_model_bullet_1, nra_model_bullet_2], 1):
    print(f"Bullet {i}")
    print(f"  Number: {b['number']}")
    print(f"  Reason: {b['reason']}")
    print(f"  Action: {b['action']}")

Bullet 1
  Number: Plain model accuracy: 0.730, Balanced model accuracy: 0.500, Baseline: 0.730
  Reason: The generator defines `resolved = random.random() < 0.72` – the label is independent of any feature. No model can learn a real signal.
  Action: Do not use this ML classifier for any business decision. Instead, rely on the RAG layer for case-specific, grounded answers.
Bullet 2
  Number: Balanced confusion matrix: TN=11, FP=16, FN=34, TP=39
  Reason: `class_weight='balanced'` forces the model to predict the minority class (unresolved) more often; because the label is purely random, this shift trades correct majority guesses for wrong minority predictions – hence accuracy drops to 0.5.
  Action: If this model is revisited with better features, set a minimum performance requirement: it must beat the baseline accuracy by at least 0.05 on a validation set. This threshold will be reviewed by the data science lead on Day 196 and re‑evaluated quarterly if new feature engineering is perfor

### Task 3 — RAG Layer: FAISS Retrieval over Resolved Complaints (20 pts)

Build the knowledge base from **resolved complaints only** (`df_clean[df_clean['resolved']==True]`)
— these are the only rows with a known outcome (`days_to_resolve`, `satisfaction_score`) worth
retrieving as precedent. Vectorize with the **same `TfidfVectorizer` pattern from Month 9**, index
with FAISS, and implement a retrieval function.


In [4]:
# TASK 3: RAG Retrieval Layer (CORRECTED)
# Goal: build a FAISS index over resolved complaints for retrieval.
# Method: install faiss if missing, filter to resolved-only, TF-IDF vectorize,
#         L2-normalize, create IndexFlatIP, and implement a retrieval function.

import subprocess
import sys

# Ensure faiss is installed
try:
    import faiss
except ImportError:
    print("Installing faiss-cpu...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "faiss-cpu"])
    import faiss

import numpy as np
import pandas as pd

# 1. Filter resolved complaints
resolved_df = df_clean[df_clean["resolved"] == True].reset_index(drop=True)
print(f"Knowledge base size (resolved complaints only): {len(resolved_df)}")

# 2. TF-IDF vectorizer for the knowledge base
kb_tfidf = TfidfVectorizer(max_features=200, stop_words="english")
kb_vectors = kb_tfidf.fit_transform(resolved_df["complaint_text"]).toarray().astype("float32")

# 3. Normalize for cosine similarity
faiss.normalize_L2(kb_vectors)

# 4. Build FAISS index (Inner Product = cosine after normalization)
index = faiss.IndexFlatIP(kb_vectors.shape[1])
index.add(kb_vectors)
print(f"FAISS index vector dim: {kb_vectors.shape[1]}, total vectors: {index.ntotal}")

def retrieve_similar_complaints(query, k=3):
    """Return the top-k most similar resolved complaints with their outcomes."""
    query_vec = kb_tfidf.transform([query]).toarray().astype("float32")
    faiss.normalize_L2(query_vec)
    scores, indices = index.search(query_vec, k)

    results = []
    for score, idx in zip(scores[0], indices[0]):
        if idx < len(resolved_df):  # ensure valid index
            row = resolved_df.iloc[idx]
            results.append({
                "complaint_text": row["complaint_text"],
                "days_to_resolve": row["days_to_resolve"],
                "satisfaction_score": row["satisfaction_score"],
                "similarity": float(score)
            })
    return results

# Define sample_query so it can be used in Task 4
sample_query = "Internet speed far below promised Broadband plan speed during evening hours"

# Test retrieval
results = retrieve_similar_complaints(sample_query, k=3)
print("Sample query results:")
for r in results:
    print(r)

Installing faiss-cpu...
Knowledge base size (resolved complaints only): 294
FAISS index vector dim: 65, total vectors: 294
Sample query results:
{'complaint_text': 'Internet speed far below promised Broadband plan speed during evening hours', 'days_to_resolve': np.float64(2.0), 'satisfaction_score': np.float64(5.0), 'similarity': 1.0000001192092896}
{'complaint_text': 'Internet speed far below promised Broadband plan speed during evening hours', 'days_to_resolve': np.float64(0.0), 'satisfaction_score': np.float64(5.0), 'similarity': 1.0000001192092896}
{'complaint_text': 'Internet speed far below promised Broadband plan speed during evening hours', 'days_to_resolve': np.float64(4.0), 'satisfaction_score': np.float64(5.0), 'similarity': 1.0000001192092896}


### Task 4 — RAG Generation Layer: Groq-Backed Resolution Suggestion (20 pts)

Wire Task 3's retrieval into an actual generation step using the **pinned Month 10 LangChain +
Groq stack**. This is the part that turns "search" into "RAG" — the retrieved precedents become
context in a prompt, not just search results shown to a human.

```python
from google.colab import userdata
import os
os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
from langchain_groq import ChatGroq
```

**Note:** the LLM call itself is non-deterministic and needs your own Groq key — it is not scored
on exact wording. It IS scored on: retrieved context correctly injected into the prompt, prompt
explicitly grounding the model in the retrieved precedents (not asking it to freelance), and the
function returning both the answer and which complaints it was grounded in (traceability).


In [5]:
# TASK 4: RAG Generation with Groq (CORRECTED)
# Goal: use LangChain + Groq to generate a resolution suggestion grounded in retrieved precedents.
# Method: install missing packages, ensure sample_query exists, build prompt with context.

import subprocess
import sys

# Install langchain-groq if missing
try:
    from langchain_groq import ChatGroq
except ImportError:
    print("Installing langchain-groq...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "langchain-groq"])
    from langchain_groq import ChatGroq

from google.colab import userdata
import os
os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

def suggest_resolution(query, k=3):
    # 1. Retrieve similar complaints
    retrieved = retrieve_similar_complaints(query, k)

    # 2. Build context block
    context_lines = []
    for i, r in enumerate(retrieved, 1):
        line = (f"{i}. Complaint: \"{r['complaint_text']}\" | "
                f"Days to resolve: {r['days_to_resolve']} | "
                f"Satisfaction: {r['satisfaction_score']}")
        context_lines.append(line)
    context_block = "\n".join(context_lines)

    # 3. Prompt – explicitly grounds the model
    prompt = f"""
You are a customer support assistant for a telecom company.
You are given a new customer complaint and a set of similar past complaints that were resolved.

Your task: suggest a resolution for the new complaint **based only on the past complaints provided below**.
Do not use general telecom knowledge outside of these examples.

New complaint: "{query}"

Similar past resolved complaints:
{context_block}

Based on the above precedents, what resolution would you suggest and why?
"""

    # 4. LLM call
    llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0.2)
    response = llm.invoke(prompt)

    return {
        "suggested_resolution": response.content,
        "grounded_in": retrieved,
    }

# Use the sample_query defined in Task 3
if 'sample_query' not in globals():
    sample_query = "Internet speed far below promised Broadband plan speed during evening hours"

result = suggest_resolution(sample_query, k=3)
print("Suggested resolution:", result["suggested_resolution"])
print("\nGrounded in these precedents:")
for r in result["grounded_in"]:
    print(f" - {r['complaint_text'][:60]}... (similarity: {r['similarity']:.3f})")

Installing langchain-groq...
Suggested resolution: Based on the past resolved complaints, I would suggest the following resolution for the new complaint:

**Resolution:** "We will investigate and upgrade your internet speed to meet the promised Broadband plan speed during evening hours. If the issue persists, we will provide a complimentary data boost for the next 30 days."

**Why:**

- All three past complaints were resolved with a satisfaction rating of 5.0, indicating that the resolution was effective in addressing the customer's issue.
- Two out of three past complaints were resolved in 0.0 and 2.0 days, which suggests that the issue can be resolved quickly.
- The third complaint took 4.0 days to resolve, but still resulted in a satisfaction rating of 5.0, indicating that the resolution was effective even if it took a bit longer.

Given this pattern, it is likely that the issue can be resolved quickly and effectively by upgrading the internet speed to meet the promised Broadband pl

### Task 5 — NRA: Model + RAG Layer Status Report (10 pts)

One NRA bullet summarizing today's two layers together, feeding directly into Day 195/196's
final report and README.


In [9]:
# TASK 5: NRA Layer Status Report (CORRECTED)
# Goal: one bullet summarizing today's ML and RAG layers.
# Method: use numbers from earlier tasks (ensure variables exist).

# Ensure variables exist from Task 1 and Task 3
if 'plain_acc' not in globals():
    plain_acc = accuracy_score(y_test, y_pred_plain)
if 'baseline_acc' not in globals():
    baseline_acc = max(y_test.mean(), 1 - y_test.mean())
if 'resolved_df' not in globals():
    # In case Task 3 didn't run, compute it now
    resolved_df = df_clean[df_clean["resolved"] == True].reset_index(drop=True)

nra_layer_report = {
    "number": f"ML model accuracy {plain_acc:.3f} vs baseline {baseline_acc:.3f}; FAISS knowledge base of {len(resolved_df)} resolved complaints built.",
    "reason": "The ML model fails because the target 'resolved' is a pure random draw (p=0.72) – no feature has any predictive power. In contrast, RAG retrieval relies only on the similarity of complaint text, which does not require any label to be learnable; it simply retrieves known outcomes from comparable past cases. This structural difference means one layer can provide useful information while the other cannot.",
    "action": "In the Day 195 Streamlit app, the primary feature will be the RAG-based resolution suggester; the ML classifier will be displayed only as a supplementary metric with a clear warning about its unreliability."
}

print(f"Number: {nra_layer_report['number']}")
print(f"Reason: {nra_layer_report['reason']}")
print(f"Action: {nra_layer_report['action']}")

Number: ML model accuracy 0.730 vs baseline 0.730; FAISS knowledge base of 294 resolved complaints built.
Reason: The ML model fails because the target 'resolved' is a pure random draw (p=0.72) – no feature has any predictive power. In contrast, RAG retrieval relies only on the similarity of complaint text, which does not require any label to be learnable; it simply retrieves known outcomes from comparable past cases. This structural difference means one layer can provide useful information while the other cannot.
Action: In the Day 195 Streamlit app, the primary feature will be the RAG-based resolution suggester; the ML classifier will be displayed only as a supplementary metric with a clear warning about its unreliability.


### Bonus (10★) — Persist Layer 3 + Layer 4 Artifacts

Day 195/196 need to load today's model and index without retraining. Save:
- The fitted `tfidf`, `scaler`, and `clf_plain` (or `clf_balanced` — pick the one Task 2's Action
  actually recommends) via `joblib`
- The FAISS index via `faiss.write_index`
- `kb_tfidf` via `joblib`, and `resolved_df` (the metadata index maps back to) via CSV


In [7]:
# BONUS: Persist artifacts (CORRECTED)
# Goal: save all trained objects and metadata for Day 195.
# Method: use joblib, ensure all objects exist, handle missing imports.

import joblib
import os

# Ensure faiss is available
try:
    import faiss
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "faiss-cpu"])
    import faiss

# Ensure all variables exist; if not, raise informative errors
required_objs = {
    "tfidf": tfidf,
    "scaler": scaler,
    "clf_plain": clf_plain,
    "index": index,
    "kb_tfidf": kb_tfidf,
    "resolved_df": resolved_df
}
for name, obj in required_objs.items():
    if obj is None:
        raise NameError(f"Variable '{name}' is not defined. Run Tasks 1 and 3 first.")

joblib.dump(tfidf, "layer3_tfidf.joblib")
joblib.dump(scaler, "layer3_scaler.joblib")
joblib.dump(clf_plain, "layer3_model.joblib")   # as recommended in Task 2

faiss.write_index(index, "layer4_faiss.index")
joblib.dump(kb_tfidf, "layer4_kb_tfidf.joblib")
resolved_df.to_csv("layer4_resolved_kb.csv", index=False)

print("Layer 3 + Layer 4 artifacts saved successfully.")

Layer 3 + Layer 4 artifacts saved successfully.


---
## Section 3 — Scoring Rubric

| Task | Points | Criteria |
|---|---|---|
| T1 — ML Feature Layer | 20 | TF-IDF + numeric features combined correctly; both models trained; all 5 metrics printed from real output |
| T2 — NRA Model Business Case | 20 | Reason verified against the actual `resolved = random.random() < 0.72` line, not invented; Action names what NOT to build and what replaces it |
| T3 — RAG Retrieval Layer | 20 | Knowledge base correctly filtered to resolved-only; FAISS index built with L2-normalized vectors; retrieval function returns correct structure |
| T4 — RAG Generation Layer | 20 | Prompt explicitly grounds the LLM in retrieved context only; function returns both answer and precedents (traceability) |
| T5 — NRA Layer Status Report | 10 | One bullet, Reason stops at mechanism (no bleed into Action) |
| **Total** | **90** | |
| Bonus — Persist artifacts | 10★ | All 6 artifacts saved with correct filenames for Day 195 to load |

**Common deduction patterns to watch for (per the NRA framework):**
- Reason that restates the Number instead of explaining *why* the balanced model underperforms
- Reason that drifts into Action ("...so we should use RAG instead") — that belongs in Action only
- Fabricating a text-based cause for `resolved` instead of checking the generator line
- Building the knowledge base from the full 400 rows instead of the 294 resolved-only rows
